# DBSCAN Clustering

## Learning Objectives
By the end of this notebook, you will:
- Understand density-based clustering concepts
- Know when DBSCAN is better than K-Means
- Master the eps and min_samples parameters
- Use the k-distance graph to find optimal eps
- Handle outliers/noise points with DBSCAN

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN, KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_moons, make_blobs
from sklearn.neighbors import NearestNeighbors
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

## 1. Why DBSCAN?

### K-Means Limitations
K-Means works well for **spherical, evenly-sized clusters**. But real data often has:
- Arbitrary-shaped clusters
- Clusters of different sizes/densities
- Outliers/noise

Let's see where K-Means fails:

In [ ]:
# Create moon-shaped data (non-spherical clusters)
X_moons, y_moons = make_moons(n_samples=300, noise=0.05, random_state=42)

# Try K-Means
kmeans = KMeans(n_clusters=2, random_state=42)
kmeans_labels = kmeans.fit_predict(X_moons)

# Try DBSCAN
dbscan = DBSCAN(eps=0.2, min_samples=5)
dbscan_labels = dbscan.fit_predict(X_moons)

# Compare
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, cmap='viridis')
axes[0].set_title('True Clusters', fontsize=12)

axes[1].scatter(X_moons[:, 0], X_moons[:, 1], c=kmeans_labels, cmap='viridis')
axes[1].set_title('K-Means (FAILS!)', fontsize=12)

axes[2].scatter(X_moons[:, 0], X_moons[:, 1], c=dbscan_labels, cmap='viridis')
axes[2].set_title('DBSCAN (SUCCESS!)', fontsize=12)

plt.suptitle('K-Means vs DBSCAN on Non-Spherical Data', fontsize=14)
plt.tight_layout()
plt.show()

## 2. How DBSCAN Works

### Core Concepts

DBSCAN groups points that are **densely packed together**, marking points in low-density regions as outliers.

### Key Parameters

| Parameter | Description |
|-----------|-------------|
| **eps** (ε) | Maximum distance between two points to be considered neighbors |
| **min_samples** | Minimum points required to form a dense region (core point) |

### Point Types

1. **Core Point**: Has at least `min_samples` neighbors within `eps` distance
2. **Border Point**: Within `eps` of a core point but has fewer than `min_samples` neighbors
3. **Noise Point**: Neither core nor border (labeled as -1)

In [ ]:
# Visualize core, border, and noise points
np.random.seed(42)
X_demo = np.vstack([
    np.random.randn(50, 2) * 0.5 + [0, 0],
    np.random.randn(50, 2) * 0.5 + [3, 3],
    np.random.randn(5, 2) * 0.3 + [1.5, 1.5]  # Noise/outliers
])

dbscan_demo = DBSCAN(eps=0.5, min_samples=5)
labels_demo = dbscan_demo.fit_predict(X_demo)

# Get core sample indices
core_samples = dbscan_demo.core_sample_indices_

plt.figure(figsize=(10, 6))

# Plot all points
colors = ['red' if l == -1 else 'blue' if l == 0 else 'green' for l in labels_demo]
plt.scatter(X_demo[:, 0], X_demo[:, 1], c=colors, alpha=0.6, s=50)

# Highlight core points
plt.scatter(X_demo[core_samples, 0], X_demo[core_samples, 1], 
            c='none', edgecolors='black', s=150, linewidths=2, label='Core Points')

# Mark noise points
noise_mask = labels_demo == -1
plt.scatter(X_demo[noise_mask, 0], X_demo[noise_mask, 1], 
            c='red', marker='x', s=100, label='Noise Points')

plt.title('DBSCAN: Core Points (circled) and Noise Points (X)', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Number of clusters found: {len(set(labels_demo)) - (1 if -1 in labels_demo else 0)}")
print(f"Number of noise points: {sum(labels_demo == -1)}")

## 3. Finding Optimal eps: The K-Distance Graph

The most challenging part of DBSCAN is choosing the right `eps`. The **k-distance graph** helps:

1. For each point, calculate distance to its k-th nearest neighbor (k = min_samples)
2. Sort these distances
3. Plot them
4. Look for the "elbow" - where the curve bends sharply

In [ ]:
def find_optimal_eps(X, min_samples=5):
    """
    Use k-distance graph to find optimal eps.
    
    Parameters:
    -----------
    X : array-like
        Feature matrix
    min_samples : int
        min_samples parameter for DBSCAN
    
    Returns:
    --------
    distances : array
        Sorted k-distances
    """
    # Calculate k-nearest neighbors
    neighbors = NearestNeighbors(n_neighbors=min_samples)
    neighbors.fit(X)
    distances, indices = neighbors.kneighbors(X)
    
    # Get distance to k-th neighbor (last column)
    k_distances = distances[:, -1]
    
    # Sort in ascending order
    k_distances_sorted = np.sort(k_distances)
    
    return k_distances_sorted

# Apply to our demo data
k_distances = find_optimal_eps(X_demo, min_samples=5)

plt.figure(figsize=(10, 6))
plt.plot(range(len(k_distances)), k_distances, 'b-')
plt.axhline(y=0.5, color='r', linestyle='--', label='Chosen eps=0.5')
plt.xlabel('Points sorted by distance')
plt.ylabel('k-distance (k=5)')
plt.title('K-Distance Graph for Finding Optimal eps')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("Look for the 'elbow' - where the curve starts to rise sharply.")
print("Points after the elbow are likely outliers.")

## 4. Parameter Sensitivity

Let's see how different eps values affect clustering:

In [ ]:
# Test different eps values
eps_values = [0.1, 0.3, 0.5, 0.8, 1.0]

fig, axes = plt.subplots(1, 5, figsize=(20, 4))

for ax, eps in zip(axes, eps_values):
    dbscan = DBSCAN(eps=eps, min_samples=5)
    labels = dbscan.fit_predict(X_demo)
    
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = sum(labels == -1)
    
    colors = ['gray' if l == -1 else plt.cm.viridis(l / max(1, max(labels))) for l in labels]
    ax.scatter(X_demo[:, 0], X_demo[:, 1], c=colors)
    ax.set_title(f'eps={eps}\nClusters: {n_clusters}, Noise: {n_noise}')

plt.suptitle('Effect of eps on DBSCAN Clustering', fontsize=14)
plt.tight_layout()
plt.show()

## 5. DBSCAN vs K-Means: When to Use Each

| Aspect | K-Means | DBSCAN |
|--------|---------|--------|
| **Cluster Shape** | Spherical only | Any shape |
| **Number of Clusters** | Must specify K | Auto-detected |
| **Outlier Handling** | Assigns to nearest cluster | Labels as noise (-1) |
| **Cluster Size** | Similar sizes work best | Handles varying sizes |
| **Speed** | Fast (O(n)) | Slower (O(n²) or O(n log n)) |
| **Parameters** | K | eps, min_samples |

In [ ]:
# Create data with clusters of different densities
np.random.seed(42)
X_varied = np.vstack([
    np.random.randn(200, 2) * 0.3 + [0, 0],      # Dense cluster
    np.random.randn(50, 2) * 1.0 + [4, 4],       # Sparse cluster
    np.random.randn(100, 2) * 0.5 + [0, 4]       # Medium cluster
])

# Scale the data
scaler = StandardScaler()
X_varied_scaled = scaler.fit_transform(X_varied)

# Compare algorithms
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# K-Means
kmeans = KMeans(n_clusters=3, random_state=42)
kmeans_labels = kmeans.fit_predict(X_varied_scaled)
axes[0].scatter(X_varied_scaled[:, 0], X_varied_scaled[:, 1], c=kmeans_labels, cmap='viridis')
axes[0].set_title('K-Means (K=3)\nForces 3 clusters', fontsize=12)

# DBSCAN
dbscan = DBSCAN(eps=0.3, min_samples=5)
dbscan_labels = dbscan.fit_predict(X_varied_scaled)
n_clusters = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
axes[1].scatter(X_varied_scaled[:, 0], X_varied_scaled[:, 1], c=dbscan_labels, cmap='viridis')
axes[1].set_title(f'DBSCAN\nFound {n_clusters} clusters + noise', fontsize=12)

plt.suptitle('K-Means vs DBSCAN on Varied Density Data', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Practical Example: Customer Data with Outliers

In [ ]:
# Create customer-like data with outliers
np.random.seed(42)

# Normal customers
normal_customers = np.vstack([
    np.random.randn(100, 2) * [5000, 5] + [30000, 25],   # Low income, young
    np.random.randn(100, 2) * [8000, 8] + [70000, 45],   # High income, middle-aged
    np.random.randn(80, 2) * [10000, 5] + [50000, 60]    # Medium income, older
])

# Outliers (unusual spending patterns)
outliers = np.array([
    [150000, 30],  # Very high income young person
    [20000, 80],   # Low income elderly
    [200000, 50],  # Extremely high income
])

X_customers = np.vstack([normal_customers, outliers])

# Scale
scaler = StandardScaler()
X_customers_scaled = scaler.fit_transform(X_customers)

# Find optimal eps
k_distances = find_optimal_eps(X_customers_scaled, min_samples=5)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# K-distance graph
axes[0].plot(k_distances)
axes[0].axhline(y=0.5, color='r', linestyle='--', label='eps=0.5')
axes[0].set_xlabel('Points')
axes[0].set_ylabel('k-distance')
axes[0].set_title('K-Distance Graph')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# DBSCAN result
dbscan = DBSCAN(eps=0.5, min_samples=5)
labels = dbscan.fit_predict(X_customers_scaled)

# Plot with original scale
scatter = axes[1].scatter(X_customers[:, 0], X_customers[:, 1], c=labels, cmap='viridis')
axes[1].set_xlabel('Income')
axes[1].set_ylabel('Age')
axes[1].set_title(f'Customer Segments (Noise points in purple)')

# Mark outliers
outlier_mask = labels == -1
axes[1].scatter(X_customers[outlier_mask, 0], X_customers[outlier_mask, 1], 
                c='red', marker='x', s=200, label='Outliers')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"\nClusters found: {len(set(labels)) - (1 if -1 in labels else 0)}")
print(f"Outliers detected: {sum(labels == -1)}")

## 7. Exercise: Apply DBSCAN to the Seeds Dataset

1. Load the seeds dataset
2. Scale the features
3. Use k-distance graph to find optimal eps
4. Apply DBSCAN
5. Compare results with K-Means

In [ ]:
# Load data
seeds = pd.read_csv('../../00-data/seeds.csv')
X_seeds = seeds.values

# Scale
scaler = StandardScaler()
X_seeds_scaled = scaler.fit_transform(X_seeds)

# YOUR CODE HERE
# 1. Plot k-distance graph to find optimal eps
# 2. Apply DBSCAN with your chosen eps and min_samples=5
# 3. Compare with K-Means (K=3)
# 4. Visualize using two features (e.g., area vs perimeter)



## Summary

### Key Takeaways:

1. **DBSCAN** finds clusters based on density, not distance to centroids
2. **eps** defines the neighborhood radius
3. **min_samples** defines the minimum density for a core point
4. Use **k-distance graph** to find optimal eps
5. DBSCAN naturally identifies **outliers** (labeled as -1)
6. DBSCAN doesn't require specifying the number of clusters

### When to Use DBSCAN:
- Arbitrary-shaped clusters
- Unknown number of clusters
- Data contains outliers
- Clusters have varying densities

---

**Next:** [04_Cluster_Evaluation_Metrics.ipynb](04_Cluster_Evaluation_Metrics.ipynb)